# 🧪 The Tyro Wash Test — Track A starter

**Team Dark.Tyro · FIT5230 Theme 2 (Text-to-Image), Dark side**

> **Your task: protect 10 faces so our purifier cannot wash the protection off —
> without wrecking the photographs.**

## You only edit one function

Everything in this notebook is scaffolding except **`protect()` in Section 3**. Change that one
function and you have an entry. You never need to read our attack code, and you do not have to
tell us how your shield works — our purifier never inspects the perturbation.

    10 clean PNGs  ->  your protect()  ->  10 protected PNGs  ->  send them to us

## Cost, honestly

| route | hardware | time for all 10 |
|---|---|---|
| **A · no shield** (see the interface work) | anything | **seconds** |
| **B · noise shield** (no GPU at all) | anything | **seconds** |
| **C · PhotoGuard encoder attack** (the real thing, included below) | free Colab T4 | **≈ 5 minutes** |

If you have never written a defence, run route C unchanged: **that already is a valid entry.**
Then change one number and it is your entry.

## How you are scored — two axes, both published

| axis | meaning | you want |
|---|---|---|
| **`R_pipe`** | how much of the editing pipeline our wash restored. 1.0 = we stripped you completely, 0 = your shield held | **low** |
| **LPIPS(protected, clean)** | how much of the photograph you destroyed to get there | **low**, and **≤ 0.10** to be ranked |

Two axes, because with robustness alone the winning move is trivial: perturb until the picture
is noise. Nothing can be washed off a ruined photo — but nothing can be **edited** either, so
that "defence" defeats its own purpose. It lands in the top-right corner of the plot, in public.

**One honest guard rail.** If your shield does not measurably disturb the edit, `R_pipe` becomes
a ratio of two near-zero numbers. We then report **"shield did not engage"** and tell you what we
measured, instead of printing a flattering number. We run the same check on our own shield first.


---
## 1 · Setup

Colab: **Runtime → Change runtime type → T4 GPU** for route C. Routes A and B need no GPU.


In [2]:
#@title Install + imports  (~40 s on a fresh Colab)
import subprocess, sys, importlib.util, os, time, pathlib, zipfile
for mod, pkg in {'torch':'torch', 'diffusers':'diffusers', 'lpips':'lpips',
                 'skimage':'scikit-image'}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.run(f'{sys.executable} -m pip install -q {pkg}', shell=True, check=False)

import torch, numpy as np
from PIL import Image

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU only')
if DEV == 'cpu':
    print('  -> routes A and B work fine. Route C needs a GPU (Runtime > Change runtime type).')


device: cpu | CPU only
  -> routes A and B work fine. Route C needs a GPU (Runtime > Change runtime type).


In [3]:
#@title Point at the image pack
# Unzip tyro_wash_test_trackA.zip next to this notebook, or set PACK to wherever it landed.
PACK = pathlib.Path('tyro_wash_test_trackA')      # <-- edit if needed
OUT  = pathlib.Path('my_submission')

assert (PACK / 'clean').is_dir(), f'no clean/ folder under {PACK.resolve()}'
NAMES = sorted(os.listdir(PACK / 'clean'))
OUT.mkdir(exist_ok=True)
print(f'{len(NAMES)} images in {PACK}/clean, masks in {PACK}/mask')
print('output ->', OUT.resolve())


10 images in tyro_wash_test_trackA/clean, masks in tyro_wash_test_trackA/mask
output -> /Users/ez_us/Documents/5230/5230-Assignment/M2/challenge/my_submission


---
## 2 · The reference shield (optional reading)

PhotoGuard's **encoder attack**. Stable Diffusion does not work on pixels — it first squeezes
the image through a VAE encoder into a small latent code, and edits *that*. So the cheapest way
to sabotage an edit is to make the encoder read the wrong thing.

> The editor never sees your photograph. It sees a **summary** of it. This attack rewrites the
> summary — nudging pixels, invisibly, until the encoder describes a flat grey rectangle instead
> of a face. The editor then confidently repaints something that was never there.

It is projected gradient descent with the picture held inside an `eps`-sized box, so the change
stays imperceptible. **`eps` is your main dial.** Bigger = harder to wash off, and more visible.


In [4]:
#@title The PhotoGuard encoder attack (route C). Loads a VAE once, ~10 s per image after that.
_vae = None
def _get_vae():
    global _vae
    if _vae is None:
        from diffusers import AutoencoderKL
        for mid in ['runwayml/stable-diffusion-inpainting',
                    'stable-diffusion-v1-5/stable-diffusion-inpainting',
                    'stabilityai/sd-vae-ft-mse']:
            try:
                _vae = AutoencoderKL.from_pretrained(mid, subfolder=None if 'vae' in mid else 'vae')
                print('vae from', mid); break
            except Exception:
                continue
        assert _vae is not None, 'could not load a VAE'
        _vae = _vae.to(DEV).eval().requires_grad_(False)
    return _vae

def photoguard_encoder(img, eps_levels=4, step_levels=1, iters=100):
    if DEV == 'cpu' and not globals().get('ALLOW_SLOW_CPU', False):
        raise RuntimeError(
            'Route C on CPU takes about 4 MINUTES PER IMAGE (~45 min for the pack).\n'
            'On a free Colab T4 the same run is ~2 min total.\n'
            '  -> Runtime > Change runtime type > T4 GPU, then re-run from the top.\n'
            '  -> or set ALLOW_SLOW_CPU = True above if you really want to wait.')
    """PGD on the VAE encoder, dragging the latent toward the latent of flat grey.
       eps_levels / step_levels are in 0-255 grey levels, so eps=16 means 'never move a
       pixel more than 16 of 255' -- about 6%, and invisible on a photograph."""
    vae = _get_vae()
    x0  = (torch.from_numpy(np.asarray(img.convert('RGB'), np.float32))
           .permute(2, 0, 1)[None].to(DEV) / 127.5 - 1.0)
    eps, step = eps_levels / 127.5, step_levels / 127.5
    with torch.no_grad():
        target = vae.encode(torch.zeros_like(x0)).latent_dist.mean      # grey = 0 in [-1,1]
    x = x0.clone()
    for _ in range(iters):
        x.requires_grad_(True)
        loss = (vae.encode(x).latent_dist.mean - target).norm()
        g    = torch.autograd.grad(loss, x)[0]
        x    = (x.detach() - step * g.sign()).clamp(x0 - eps, x0 + eps).clamp(-1, 1)
    a = ((x[0].permute(1, 2, 0).cpu().numpy() + 1) * 127.5).round().clip(0, 255)
    return Image.fromarray(a.astype(np.uint8))


---
## 3 · ✏️ YOUR ENTRY — this is the only cell you have to change

`protect()` takes one clean 512x512 image and its mask, and returns your protected image,
same size, same mode. Nothing else in this notebook cares how you do it.

**The mask is given to you because it is given to us too** — it is an input to the editing
pipeline, not secret knowledge. White = the region the editor keeps. Black = the region it
repaints from scratch. You may use it or ignore it.

Three worked bodies are below. **Uncomment one, or write your own.**


In [5]:
#@title protect()  <-- EDIT THIS
SHIELD = 'photoguard'      #@param ['none', 'noise', 'photoguard']
EPS    = 3                 #@param {type:'integer'}   # grey levels. MEASURED on a real run:
ALLOW_SLOW_CPU = False     #@param {type:'boolean'}   #   EPS   LPIPS (mean)   ranked?
#                                                     #    3      ~0.07         yes  <- default
#                                                     #    4       0.104        NO, by 0.004
#                                                     #   16       0.415        NO, by a lot
# EPS is a per-pixel ceiling and PGD pushes most pixels to it, so cost climbs FASTER than EPS.
# Per-image LPIPS at EPS=4 ranged 0.039 to 0.152 -- a 4x spread on identical settings, so
# check the mean below rather than assuming a setting that worked on one face works on ten.

def protect(img, mask):
    """img: PIL RGB 512x512 (clean).  mask: PIL RGB 512x512.  returns: PIL RGB 512x512."""

    # --- route A: no shield. The control. Shows you the interface works. ---
    if SHIELD == 'none':
        return img

    # --- route B: a noise shield. No GPU, no model, ~0 s. A real (weak) baseline. ---
    # sigma = EPS/8 is not arbitrary: at EPS=16 it costs LPIPS ~0.019, the SAME perceptual
    # price as the PhotoGuard reference below. So B vs C is a fair fight, and it asks a real
    # question -- at matched cost, does STRUCTURED perturbation beat random?
    if SHIELD == 'noise':
        a = np.asarray(img, np.float32)
        a = a + np.random.default_rng(0).normal(0, EPS / 8, a.shape)
        return Image.fromarray(a.clip(0, 255).astype(np.uint8))

    # --- route C: PhotoGuard encoder attack. Needs a GPU. ~10 s/image. ---
    if SHIELD == 'photoguard':
        return photoguard_encoder(img, eps_levels=EPS)

    raise ValueError(SHIELD)

# ---------------------------------------------------------------------------
# Ideas, if you want to go past the reference shield:
#   * concentrate the budget INSIDE the white mask -- that is the only region the editor keeps
#   * attack the encoder toward a target that is not grey (another face, pure texture)
#   * make it robust to being washed: re-run the attack on a JPEG'd / blurred copy each step
#   * randomise across steps so a single fixed filter cannot subtract it
# ---------------------------------------------------------------------------
print(f'shield = {SHIELD}, eps = {EPS} levels')


shield = photoguard, eps = 16 levels


In [6]:
#@title Run it over all 10 images, and time it.
t0 = time.time()
for i, n in enumerate(NAMES, 1):
    img  = Image.open(PACK / 'clean' / n).convert('RGB')
    mask = Image.open(PACK / 'mask'  / n).convert('RGB')
    out  = protect(img, mask)
    assert out.size == (512, 512), f'{n}: protect() returned {out.size}, must be (512, 512)'
    out.convert('RGB').save(OUT / n)
    print(f'  [{i:>2}/{len(NAMES)}] {n}   {time.time()-t0:6.1f} s elapsed', flush=True)
print(f'\ndone: {len(NAMES)} images in {(time.time()-t0)/60:.1f} min -> {OUT.resolve()}')


/opt/anaconda3/envs/pyTorch/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
/opt/anaconda3/envs/pyTorch/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffus

vae from runwayml/stable-diffusion-inpainting
  [ 1/10] 1233476865_1.png    401.3 s elapsed
  [ 2/10] 1525918600_1.png    645.5 s elapsed
  [ 3/10] 178046512_1.png    892.2 s elapsed
  [ 4/10] 181707205_1.png   1134.5 s elapsed
  [ 5/10] 1961032923_1.png   1372.8 s elapsed
  [ 6/10] 2061993362_1.png   1614.2 s elapsed
  [ 7/10] 2099073485_1.png   1853.7 s elapsed
  [ 8/10] 2139626906_1.png   2096.3 s elapsed
  [ 9/10] 2167874246_1.png   2355.6 s elapsed
  [10/10] 221629697_1.png   2601.9 s elapsed

done: 10 images in 43.4 min -> /Users/ez_us/Documents/5230/5230-Assignment/M2/challenge/my_submission


---
## 4 · Self-check before you send

Same numbers we will compute. If you are over the LPIPS budget you are still plotted and
discussed — just not ranked — so it is worth knowing now rather than after the deadline.


In [7]:
#@title Perceptual cost vs the clean images
import lpips
from skimage.metrics import structural_similarity as _ss

BUDGET = 0.10
net = lpips.LPIPS(net='alex').to(DEV).eval()      # must match the scorer's net -- alex, not vgg
def _t(a): return (torch.from_numpy(a.transpose(2, 0, 1)[None]).float().to(DEV) / 127.5 - 1)

print(f'  {"image":<24}{"SSIM":>8}{"PSNR":>9}{"LPIPS":>9}')
vals = []
for n in NAMES:
    a = np.asarray(Image.open(OUT / n).convert('RGB'), np.float64)
    b = np.asarray(Image.open(PACK / 'clean' / n).convert('RGB'), np.float64)
    mse = float(((a - b) ** 2).mean())
    s   = float(_ss(a, b, channel_axis=2, data_range=255))
    p   = 99.0 if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)
    with torch.no_grad(): l = float(net(_t(a.astype(np.float32)), _t(b.astype(np.float32))).item())
    vals.append(l)
    print(f'  {n:<24}{s:>8.4f}{p:>9.2f}{l:>9.4f}')

m = float(np.mean(vals))
print(f'\n  mean LPIPS = {m:.4f}   budget = {BUDGET}')
print('  -> INSIDE the budget. You will be ranked.' if m <= BUDGET else
      '  -> OVER the budget. Plotted and discussed, but not ranked. Lower your eps.')

zp = pathlib.Path('tyro_wash_test_submission.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for n in NAMES: z.write(OUT / n, n)
print(f'\n  packed -> {zp.resolve()} ({zp.stat().st_size/1e6:.1f} MB). Send us this file.')


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /Users/ez_us/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


/opt/anaconda3/envs/pyTorch/lib/python3.13/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/pyTorch/lib/python3.13/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 233M/233M [01:51<00:00, 2.18MB/s] 


Loading model from: /opt/anaconda3/envs/pyTorch/lib/python3.13/site-packages/lpips/weights/v0.1/alex.pth
  image                       SSIM     PSNR    LPIPS
  1233476865_1.png          0.6463    27.63   0.3693
  1525918600_1.png          0.5904    28.19   0.4145
  178046512_1.png           0.6054    28.15   0.4608
  181707205_1.png           0.5950    27.83   0.3914
  1961032923_1.png          0.6145    28.48   0.5140
  2061993362_1.png          0.6338    28.31   0.4640
  2099073485_1.png          0.6087    28.17   0.3871
  2139626906_1.png          0.5239    28.17   0.4888
  2167874246_1.png          0.5510    28.53   0.5267
  221629697_1.png           0.6553    27.57   0.2866

  mean LPIPS = 0.4303   budget = 0.1
  -> OVER the budget. Plotted and discussed, but not ranked. Lower your eps.

  packed -> /Users/ez_us/Documents/5230/5230-Assignment/M2/challenge/tyro_wash_test_submission.zip (5.5 MB). Send us this file.


---
## 5 · What happens next

We run our IMPRESS-based purifier over your 10 images, then edit **clean**, **protected** and
**purified** with an identical prompt, seed and mask, and publish both axes as one scatter plot,
with your dot labelled.

**Every outcome gets published, including ours losing.** Our purifier is deliberately *blind* —
it never inspects your perturbation, so if your shield defeats it that is a real result about
IMPRESS and we will report it as one.

**Submissions close 15 September 2026.** Reply on Ed or DM us with the zip.

---

### Calibration — measured on this pack, so you can see the room you have

| shield | **LPIPS vs clean (mean of 10)** | ranked? |
|---|---|---|
| PhotoGuard encoder, `EPS = 3` *(default)* | **~0.07** | yes |
| PhotoGuard encoder, `EPS = 4` | **0.104** | no — over by 0.004 |
| PhotoGuard encoder, `EPS = 16` | 0.415 | no — far over |
| noise shield, `EPS = 16` (sigma = EPS/8) | 0.019 | yes |
| *our own purifier's output, for scale* | 0.12 | — |

**`EPS` bites harder than it looks.** It is a per-pixel ceiling and the attack pushes most pixels
to it, so cost climbs faster than `EPS` does. At `EPS = 4` the per-image LPIPS ranged from 0.039
to 0.152 — **a 4x spread at identical settings** — so a value that looked fine on one face put
the ten-image mean over the line. Start at 3 and read the budget line before you send.

**The budget is an anchor, not a cliff.** We rank on the mean of your ten. An entry a hair over
is still plotted, discussed and written up — we are not going to exclude a real defence over
0.004.

**The budget is generous.** The reference defence spends about one grey level and lands at
0.019 — five times under the cap. You may build something far more aggressive than PhotoGuard
and still be ranked. Note the last row: **our purifier damages the photograph more than the
shield it removes.** We publish that rather than hide it. If your shield forces us to spend
more still, that is a result in your favour.
